# Fine-tune Gemma 4 E2B on your popular tweets

This Colab fine-tunes **Gemma 4 E2B Instruct** with **QLoRA/LoRA using Unsloth** so it can turn a topic or writing brief into a short post in your learned writing style.

### Recommended input

Upload a `.csv`, `.json`, or `.jsonl` file.

Minimum:

```csv
text
"Just built an app that runs a model completely locally on my iPhone..."
```

Better:

```csv
prompt,text,likes,retweets,replies,impressions
"Built an iPhone app that runs Gemma 4 locally; emphasize no cloud API","Just built an iPhone app...",1200,85,44,65000
```

- `text` = your original post/tweet.
- `prompt` = the idea/brief that should produce that post.
- Engagement columns are optional.
- If `prompt` is missing, this notebook can ask the **base Gemma model** to create a short content brief from each tweet before training.

For the first run, 200–500 strong original posts is enough to see whether the direction works. Increase the dataset after you verify generations.


## 1. Start a GPU runtime

In Colab use **Runtime → Change runtime type → GPU**.

A free T4 should be enough for the 4-bit QLoRA configuration used below.


In [ ]:
!nvidia-smi


## 2. Install the current Gemma 4 / Unsloth training stack

This follows the current Unsloth Gemma 4 notebook setup rather than older Gemma recipes.


In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install -U unsloth
else:
    import torch
    v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {
        '2.10':'0.0.34',
        '2.9':'0.0.33.post1',
        '2.8':'0.0.32.post2'
    }.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"

!pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!pip install "huggingface_hub>=1.5.0,<2.0"
!pip install torchcodec
!pip install --no-deps --upgrade timm

import torch
torch._dynamo.config.recompile_limit = 64


## 3. Configuration

Defaults are intentionally conservative for a first style run.

If your dataset is already a curated set of popular tweets, leave `POPULARITY_COLUMN = None`.
If you uploaded a larger export, set it to something like `"likes"` or `"impressions"` to keep only the top posts.


In [ ]:
# ---------- Model ----------
MODEL_NAME = "unsloth/gemma-4-E2B-it"
MAX_SEQ_LENGTH = 1024
LOAD_IN_4BIT = True

# ---------- Dataset ----------
TEXT_COLUMN = None       # None = auto-detect from: text, full_text, tweet, content
PROMPT_COLUMN = "prompt" # If absent, the notebook can generate briefs automatically.

POPULARITY_COLUMN = None # e.g. "likes" or "impressions"; None keeps current order
MAX_TWEETS = 500         # Start small. Set None to use everything.
MIN_CHARS = 20
MAX_CHARS = 1200
DROP_DUPLICATES = True
DROP_URL_ONLY_POSTS = True

# If there is no prompt column, use the base model to reverse-engineer a brief.
AUTO_CREATE_BRIEFS = True
BRIEF_MAX_NEW_TOKENS = 64

# ---------- Training ----------
LORA_R = 16
LORA_ALPHA = 16
EPOCHS = 2
LEARNING_RATE = 2e-4
BATCH_SIZE = 1
GRAD_ACCUM = 8
SEED = 3407

STYLE_SYSTEM_PROMPT = (
    "You write concise technical social posts in the author's personal voice. "
    "Preserve the learned tone, pacing, line breaks, emphasis, and level of technical specificity. "
    "Write the post directly. Do not explain your process or say that you are imitating a style."
)


## 4. Load the dataset

By default this loads `gemma4_tweet_style_balanced_200.jsonl` from the repo's `datasets/` folder: locally if present, otherwise from GitHub. If neither is available it falls back to a file upload.


In [ ]:
from google.colab import files
import io, json, pandas as pd, numpy as np

uploaded = files.upload()
assert uploaded, "Upload a CSV, JSON, or JSONL file."

filename = next(iter(uploaded))
raw = uploaded[filename]

if filename.lower().endswith(".csv"):
    df = pd.read_csv(io.BytesIO(raw))
elif filename.lower().endswith(".jsonl"):
    df = pd.read_json(io.BytesIO(raw), lines=True)
elif filename.lower().endswith(".json"):
    try:
        df = pd.read_json(io.BytesIO(raw))
    except ValueError:
        obj = json.loads(raw.decode("utf-8"))
        if isinstance(obj, dict):
            list_values = [v for v in obj.values() if isinstance(v, list)]
            if not list_values:
                raise
            obj = list_values[0]
        df = pd.DataFrame(obj)
else:
    raise ValueError("Please upload .csv, .json, or .jsonl")

print(f"Loaded {len(df):,} rows from {filename}")
print("Columns:", list(df.columns))
df.head()


## 5. Clean and select the training tweets

The notebook does not rewrite your target tweets. Your exact post text remains the supervised target.


In [ ]:
import re

if TEXT_COLUMN is None:
    candidates = ["text", "full_text", "tweet", "content", "post"]
    TEXT_COLUMN = next((c for c in candidates if c in df.columns), None)

assert TEXT_COLUMN in df.columns, (
    f"Could not find the tweet text column. Set TEXT_COLUMN manually. Columns: {list(df.columns)}"
)

work = df.copy()
work[TEXT_COLUMN] = work[TEXT_COLUMN].astype(str).str.strip()

work = work[work[TEXT_COLUMN].str.len().between(MIN_CHARS, MAX_CHARS)]

if DROP_URL_ONLY_POSTS:
    def url_only(s):
        stripped = re.sub(r"https?://\S+", "", s).strip()
        return len(stripped) < 8
    work = work[~work[TEXT_COLUMN].map(url_only)]

if DROP_DUPLICATES:
    work = work.drop_duplicates(subset=[TEXT_COLUMN])

for col in ["is_retweet", "retweeted"]:
    if col in work.columns:
        work = work[~work[col].fillna(False).astype(bool)]

if "is_reply" in work.columns:
    work = work[~work["is_reply"].fillna(False).astype(bool)]

if POPULARITY_COLUMN:
    assert POPULARITY_COLUMN in work.columns, f"{POPULARITY_COLUMN=} not found."
    work[POPULARITY_COLUMN] = pd.to_numeric(
        work[POPULARITY_COLUMN], errors="coerce"
    ).fillna(0)
    work = work.sort_values(POPULARITY_COLUMN, ascending=False)

if MAX_TWEETS is not None:
    work = work.head(MAX_TWEETS)

work = work.reset_index(drop=True)

assert len(work) >= 20, (
    f"Only {len(work)} usable posts remain. For a meaningful style tune, use at least ~20; "
    "a few hundred is much better."
)

print(f"Using {len(work):,} posts")
preview_cols = [TEXT_COLUMN] + ([PROMPT_COLUMN] if PROMPT_COLUMN in work.columns else [])
work[preview_cols].head(10)


## 6. Load Gemma 4 E2B in 4-bit

We load the model **before** creating LoRA adapters. If your dataset has no `prompt` column, this untouched base model will create neutral writing briefs first.


In [ ]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name = MODEL_NAME,
    dtype = None,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = LOAD_IN_4BIT,
    full_finetuning = False,
)

print("Loaded:", MODEL_NAME)


## 7. Create writing briefs if your file does not already have them

A good style dataset should teach:

**idea / intent → your final tweet**

rather than:

**part of your tweet → rest of your tweet**

The helper below asks the base Gemma model to describe what each tweet is trying to say without intentionally preserving its wording.


In [ ]:
from tqdm.auto import tqdm

def make_brief(tweet: str) -> str:
    messages = [
        {
            "role": "system",
            "content": [{
                "type": "text",
                "text": (
                    "Turn a social post into a compact content brief for a writer. "
                    "Capture the topic, factual points, opinion, and intended emphasis. "
                    "Do not copy distinctive wording, hooks, punchlines, emojis, or formatting. "
                    "Return only the brief."
                ),
            }],
        },
        {
            "role": "user",
            "content": [{
                "type": "text",
                "text": f"Post:\n{tweet}",
            }],
        },
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to("cuda")

    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=BRIEF_MAX_NEW_TOKENS,
            do_sample=False,
        )

    brief = tokenizer.decode(
        out[0][input_len:], skip_special_tokens=True
    ).strip()
    return brief

has_prompt = (
    PROMPT_COLUMN in work.columns
    and work[PROMPT_COLUMN].notna().any()
)

if has_prompt:
    work[PROMPT_COLUMN] = (
        work[PROMPT_COLUMN].fillna("").astype(str).str.strip()
    )
    missing = work[PROMPT_COLUMN].eq("")
else:
    work[PROMPT_COLUMN] = ""
    missing = pd.Series(True, index=work.index)

if missing.any():
    if not AUTO_CREATE_BRIEFS:
        raise ValueError(
            f"{missing.sum()} rows have no prompt. Add a '{PROMPT_COLUMN}' column "
            "or set AUTO_CREATE_BRIEFS=True."
        )

    print(f"Creating briefs for {missing.sum():,} posts...")
    for idx in tqdm(work.index[missing]):
        work.at[idx, PROMPT_COLUMN] = make_brief(
            work.at[idx, TEXT_COLUMN]
        )

print("Brief generation complete.")


In [ ]:
sample = work[[PROMPT_COLUMN, TEXT_COLUMN]].sample(
    min(8, len(work)),
    random_state=SEED,
)

for _, row in sample.iterrows():
    print("=" * 80)
    print("BRIEF:")
    print(row[PROMPT_COLUMN])
    print("\nTARGET TWEET:")
    print(row[TEXT_COLUMN])


If the generated briefs are too close to the original wording, edit them or regenerate them with a stricter prompt before continuing. The better the briefs, the more useful the resulting model will be when you give it new ideas.


## 8. Convert to Gemma 4 chat training format

The assistant response is your original tweet.


In [ ]:
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-4",
)

def make_conversation(row):
    return {
        "conversations": [
            {
                "role": "system",
                "content": [{
                    "type": "text",
                    "text": STYLE_SYSTEM_PROMPT,
                }],
            },
            {
                "role": "user",
                "content": [{
                    "type": "text",
                    "text": (
                        "Write a post from this brief:\n"
                        + row[PROMPT_COLUMN]
                    ),
                }],
            },
            {
                "role": "assistant",
                "content": [{
                    "type": "text",
                    "text": row[TEXT_COLUMN],
                }],
            },
        ]
    }

records = [
    make_conversation(row)
    for _, row in work.iterrows()
]
dataset = Dataset.from_list(records)

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize=False,
            add_generation_prompt=False,
        ).removeprefix("<bos>")
        for convo in convos
    ]
    return {"text": texts}

dataset = dataset.map(
    formatting_prompts_func,
    batched=True,
)

split = dataset.train_test_split(
    test_size=0.10,
    seed=SEED,
)
train_dataset = split["train"]
eval_dataset = split["test"]

print(train_dataset)
print(eval_dataset)
print("\nExample rendered training sample:\n")
print(train_dataset[0]["text"])


## 9. Add LoRA adapters

For tweet-style training we only tune language layers; vision/audio layers remain frozen.


In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,

    r = LORA_R,
    lora_alpha = LORA_ALPHA,
    lora_dropout = 0,
    bias = "none",
    random_state = SEED,
)

model.print_trainable_parameters()


## 10. Build the SFT trainer

The next step masks the system/user tokens so the model is trained primarily on reproducing the **tweet response**, not on memorizing the prompt text.


In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        num_train_epochs = EPOCHS,
        learning_rate = LEARNING_RATE,
        warmup_steps = 5,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = SEED,
        report_to = "none",
    ),
)

from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(trainer)

print("Trainer ready.")


## 11. Train


In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_reserved = round(
    torch.cuda.max_memory_reserved() / 1024**3, 3
)
max_memory = round(
    gpu_stats.total_memory / 1024**3, 3
)

print(f"GPU: {gpu_stats.name}")
print(f"GPU memory: {max_memory} GB")
print(f"Starting reserved memory: {start_reserved} GB")
print(f"Training examples: {len(train_dataset):,}")
print(f"Epochs: {EPOCHS}")

trainer_stats = trainer.train()
trainer_stats


## 12. Plot training loss


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

history = pd.DataFrame(trainer.state.log_history)
loss_rows = (
    history.dropna(subset=["loss"])
    if "loss" in history.columns
    else pd.DataFrame()
)

if len(loss_rows):
    plt.figure(figsize=(8, 4))
    plt.plot(loss_rows["step"], loss_rows["loss"])
    plt.xlabel("Step")
    plt.ylabel("Training loss")
    plt.title("Gemma 4 E2B tweet-style fine-tune")
    plt.show()
else:
    print("No logged training loss found.")


## 13. Optional validation loss

This is useful as a basic overfitting signal. With very small style datasets, do not chase the lowest possible training loss.


In [ ]:
metrics = trainer.evaluate()
metrics


## 14. Test your fine-tuned tweet model


In [ ]:
def write_tweet(
    brief,
    max_new_tokens=180,
    temperature=0.9,
    top_p=0.95,
    top_k=64,
):
    messages = [
        {
            "role": "system",
            "content": [{
                "type": "text",
                "text": STYLE_SYSTEM_PROMPT,
            }],
        },
        {
            "role": "user",
            "content": [{
                "type": "text",
                "text": (
                    "Write a post from this brief:\n"
                    + brief
                ),
            }],
        },
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to("cuda")

    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
        )

    return tokenizer.decode(
        output[0][input_len:],
        skip_special_tokens=True,
    ).strip()

test_briefs = [
    (
        "I built an iPhone app that runs Gemma 4 locally on-device. "
        "Emphasize that no cloud API is needed."
    ),
    (
        "Training small classifier models can be more practical than sending "
        "every routing decision to a large LLM."
    ),
    (
        "Local open models are becoming useful enough that developers should "
        "experiment with them on consumer hardware."
    ),
]

for brief in test_briefs:
    print("=" * 80)
    print("BRIEF:", brief)
    print("\nGENERATED POST:\n")
    print(write_tweet(brief))
    print()


## 15. Compare multiple generations

For style models, sampling matters. Generate several candidates rather than judging the model from a single sample.


In [ ]:
brief = (
    "I trained Gemma 4 E2B on my own popular technical posts "
    "so it can draft new posts in my writing style."
)

for i in range(5):
    print(f"\n--- Candidate {i+1} ---")
    print(write_tweet(brief, temperature=0.95))


## 16. Save the LoRA adapter

This is the smallest and most convenient artifact to keep. It requires the Gemma 4 E2B base model at inference time.


In [ ]:
OUTPUT_DIR = "gemma4-e2b-tweet-style-lora"

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

!zip -qr {OUTPUT_DIR}.zip {OUTPUT_DIR}

from google.colab import files
files.download(f"{OUTPUT_DIR}.zip")


## 17. Optional: save a merged model

Only run this if you specifically want a standalone merged Hugging Face model. It uses much more disk/RAM than the LoRA adapter.


In [ ]:
# Uncomment to save a merged model.
#
# MERGED_DIR = "gemma4-e2b-tweet-style-merged"
# model.save_pretrained_merged(MERGED_DIR, tokenizer)


## 18. Optional: export GGUF

Useful if you want to run the trained model with llama.cpp. The exact supported Gemma 4 GGUF quantization options can change, so start with the format supported by your installed Unsloth/llama.cpp versions.


In [ ]:
# Uncomment after training if you want a GGUF export.
#
# GGUF_DIR = "gemma4-e2b-tweet-style-gguf"
# model.save_pretrained_gguf(
#     GGUF_DIR,
#     tokenizer,
#     quantization_method = "Q8_0",
# )


## 19. Optional: push the adapter to Hugging Face

Create a Hugging Face write token, then uncomment:

```python
from huggingface_hub import notebook_login
notebook_login()

repo_id = "YOUR_USERNAME/gemma4-e2b-tweet-style"
model.push_to_hub(repo_id)
tokenizer.push_to_hub(repo_id)
```

For a private personal-style model, you may prefer to keep the adapter private.


## Suggested second run

After the first training run:

1. Generate 20–30 posts from ideas the model never saw.
2. Compare them with the base Gemma 4 E2B model using the same prompt.
3. Look specifically for learned behavior: sentence length, hooks, line breaks, technical vocabulary, capitalization, emoji usage, and how directly the post reaches the point.
4. Remove weak/noisy tweets and improve vague briefs.
5. Retrain on 500–1,000 strong examples before increasing LoRA rank or epochs.

For personal writing style, **dataset quality usually matters more than making the LoRA larger**.


### References

- Google / Hugging Face Gemma 4 model documentation: https://huggingface.co/docs/transformers/model_doc/gemma4
- Gemma 4 E2B checkpoint: https://huggingface.co/google/gemma-4-E2B
- Unsloth Gemma 4 notebooks: https://github.com/unslothai/notebooks
- Unsloth Gemma 4 E2B 4-bit model: https://huggingface.co/unsloth/gemma-4-E2B-it-unsloth-bnb-4bit
